# PDF to Md
Bu notebook, heterojen akademik disiplinlerden (Eğitim, Hukuk, Tıp, vb.) toplanan ham PDF dokümanlarını, RAG (Retrieval-Augmented Generation) sistemleri için optimize edilmiş Markdown (MD) formatına dönüştürür.

Kullanılan Araç: `Docling (IBM Research)`

In [ ]:
!pip install docling pandas tqdm

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path
from docling.document_converter import DocumentConverter
from google.colab import files

# --- AYARLAR ---
SELECTED_CATEGORY = "klasor_adi"

input_dir = Path(f"/content/drive/MyDrive/{SELECTED_CATEGORY}")
output_dir = Path(f"/content/drive/MyDrive/{SELECTED_CATEGORY}_extracted")
output_dir.mkdir(parents=True, exist_ok=True)

def process_single_category():
    if not input_dir.exists():
        print(f"❌ Hata: /content/drive/MyDrive/{SELECTED_CATEGORY} klasörü bulunamadı! Lütfen klasörü yükleyin.")
        return

    converter = DocumentConverter()
    processed_count = 0

    print(f"{SELECTED_CATEGORY} kategorisi için işlem başlatıldı...")

    pdf_files = list(input_dir.glob("*.pdf"))
    for pdf_path in pdf_files:
        print(f"📄 İşleniyor: {pdf_path.name}")
        try:
            # Dönüştürme
            result = converter.convert(str(pdf_path))
            md_output = result.document.export_to_markdown()

            # Kaydetme
            md_name = pdf_path.stem + ".md"
            with open(output_dir / md_name, "w", encoding="utf-8") as f:
                f.write(md_output)

            processed_count += 1
        except Exception as e:
            print(f"⚠️ Hata ({pdf_path.name}): {e}")

    #Zip oluşturma ve indirme
    if processed_count > 0:
        print(f"\n✅ İşlem bitti! {processed_count} dosya hazırlandı.")
        zip_name = f"{SELECTED_CATEGORY}_processed.zip"
        os.system(f"zip -r {zip_name} {output_dir}")
        files.download(zip_name)
    else:
        print("İşlenecek PDF bulunamadı.")

process_single_category()